# Notebook 04 — Test Information, Ability Estimation & Scoring Comparison

**Goal**: Compute item/test information functions, estimate abilities (EAP), compare IRT and classical scoring methods. Per sample, real authors only.

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import girth

warnings.filterwarnings('ignore')

_MARKER = "data/processed/irt_pipeline/sample1_pretest.csv"
PROJECT_ROOT = os.path.abspath(os.getcwd())
while PROJECT_ROOT:
    if os.path.isfile(os.path.join(PROJECT_ROOT, _MARKER)):
        break
    _parent = os.path.dirname(PROJECT_ROOT)
    if _parent == PROJECT_ROOT:
        PROJECT_ROOT = os.path.abspath(os.getcwd())
        break
    PROJECT_ROOT = _parent

DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed/irt_pipeline")
IRT_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/03_irt_fitting")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/04_test_ability")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data and 2PL parameters
s1_full = pd.read_csv(os.path.join(DATA_DIR, "sample1_pretest.csv"))
s2_full = pd.read_csv(os.path.join(DATA_DIR, "sample2_validation.csv"))
s1_auth = pd.read_csv(os.path.join(DATA_DIR, "sample1_authors_only.csv")).apply(pd.to_numeric, errors='coerce')
s2_auth = pd.read_csv(os.path.join(DATA_DIR, "sample2_authors_only.csv")).apply(pd.to_numeric, errors='coerce')
meta = pd.read_csv(os.path.join(DATA_DIR, "item_metadata.csv"))

p1 = np.load(os.path.join(IRT_DIR, 'irt_2pl_params_sample1.npz'))
p2 = np.load(os.path.join(IRT_DIR, 'irt_2pl_params_sample2.npz'))
a1, b1 = p1['a'], p1['b']
a2, b2 = p2['a'], p2['b']

author_cols = s1_auth.columns.tolist()
DEMO_COLS = ['Submited', 'age', 'sex ', 'humanities or not', 'education and profession']
foil_cols = meta.loc[meta['is_foil'] == True, 'item_label'].tolist()
foil_cols = [c for c in foil_cols if c in s1_full.columns]

# Coerce to numeric
for col in author_cols + foil_cols:
    s1_full[col] = pd.to_numeric(s1_full[col], errors='coerce')
    s2_full[col] = pd.to_numeric(s2_full[col], errors='coerce')

X1 = s1_auth.dropna().values.astype(int)
X2 = s2_auth.dropna().values.astype(int)
print(f"Sample 1: {X1.shape}, Sample 2: {X2.shape}")
print(f"2PL params loaded: {len(a1)} items")

Sample 1: (449, 101), Sample 2: (798, 101)
2PL params loaded: 101 items


---
## 4a. Item Information Functions (IIFs)

In [2]:
theta_range = np.linspace(-4, 4, 200)

def item_information(a, b, theta):
    """I(theta) = a^2 * P(theta) * Q(theta) for 2PL."""
    P = 1.0 / (1.0 + np.exp(-a * (theta - b)))
    return a**2 * P * (1 - P)

# Compute IIFs for all items
iif1 = np.array([item_information(a1[j], b1[j], theta_range) for j in range(len(a1))])
iif2 = np.array([item_information(a2[j], b2[j], theta_range) for j in range(len(a2))])

# Peak information per item
peak1 = iif1.max(axis=1)
peak_theta1 = theta_range[iif1.argmax(axis=1)]
peak2 = iif2.max(axis=1)
peak_theta2 = theta_range[iif2.argmax(axis=1)]

print("Item Information Summary:")
print(f"  Sample 1: mean peak = {peak1.mean():.3f}, mean peak theta = {peak_theta1.mean():.3f}")
print(f"  Sample 2: mean peak = {peak2.mean():.3f}, mean peak theta = {peak_theta2.mean():.3f}")

Item Information Summary:
  Sample 1: mean peak = 0.835, mean peak theta = 0.022
  Sample 2: mean peak = 0.752, mean peak theta = 0.841


---
## 4b. Test Information Function (TIF) and SEM

In [3]:
# TIF = sum of IIFs
tif1 = iif1.sum(axis=0)
tif2 = iif2.sum(axis=0)

# SEM = 1/sqrt(TIF)
sem1 = 1.0 / np.sqrt(np.maximum(tif1, 1e-10))
sem2 = 1.0 / np.sqrt(np.maximum(tif2, 1e-10))

# Conditional reliability r(theta) = 1 - 1/TIF(theta)
rel1 = 1 - 1.0 / np.maximum(tif1, 1.0)
rel2 = 1 - 1.0 / np.maximum(tif2, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# TIF
axes[0].plot(theta_range, tif1, 'b-', linewidth=2, label='Sample 1')
axes[0].plot(theta_range, tif2, 'r-', linewidth=2, label='Sample 2')
axes[0].set_xlabel('θ')
axes[0].set_ylabel('Information')
axes[0].set_title('Test Information Function')
axes[0].legend()
axes[0].axhline(5, color='gray', linestyle='--', alpha=0.5, label='I=5')

# SEM
axes[1].plot(theta_range, sem1, 'b-', linewidth=2, label='Sample 1')
axes[1].plot(theta_range, sem2, 'r-', linewidth=2, label='Sample 2')
axes[1].set_xlabel('θ')
axes[1].set_ylabel('SEM')
axes[1].set_title('Standard Error of Measurement')
axes[1].legend()

# Conditional reliability
axes[2].plot(theta_range, rel1, 'b-', linewidth=2, label='Sample 1')
axes[2].plot(theta_range, rel2, 'r-', linewidth=2, label='Sample 2')
axes[2].set_xlabel('θ')
axes[2].set_ylabel('Reliability')
axes[2].set_title('Conditional Reliability')
axes[2].legend()
axes[2].axhline(0.7, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tif_sem_reliability.png'), dpi=150)
plt.show()

for name, tif, sem in [('Sample 1', tif1, sem1), ('Sample 2', tif2, sem2)]:
    peak_idx = np.argmax(tif)
    reliable_range = theta_range[tif > 5]
    print(f"\n{name}:")
    print(f"  Peak TIF = {tif.max():.1f} at theta = {theta_range[peak_idx]:.2f}")
    print(f"  Min SEM = {sem.min():.3f} at peak")
    if len(reliable_range) > 0:
        print(f"  Reliable range (TIF > 5): [{reliable_range.min():.2f}, {reliable_range.max():.2f}]")
    else:
        print(f"  TIF never exceeds 5")


Sample 1:
  Peak TIF = 53.0 at theta = -0.14
  Min SEM = 0.137 at peak
  Reliable range (TIF > 5): [-3.56, 2.63]

Sample 2:
  Peak TIF = 42.6 at theta = 0.02
  Min SEM = 0.153 at peak
  Reliable range (TIF > 5): [-3.24, 2.75]


---
## 4c. Ability Estimation (EAP)

In [4]:
theta1_est = girth.ability_eap(X1.T, b1, a1)
theta2_est = girth.ability_eap(X2.T, b2, a2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (theta, name, color) in enumerate([
    (theta1_est, 'Sample 1', 'steelblue'),
    (theta2_est, 'Sample 2', 'darkorange')
]):
    axes[i, 0].hist(theta, bins=40, edgecolor='black', alpha=0.7, color=color, density=True)
    x_norm = np.linspace(theta.min(), theta.max(), 100)
    axes[i, 0].plot(x_norm, stats.norm.pdf(x_norm, theta.mean(), theta.std()),
                    'r-', linewidth=2, label='Normal fit')
    axes[i, 0].set_title(f'{name}: Theta Distribution')
    axes[i, 0].set_xlabel('θ (EAP)')
    axes[i, 0].legend()
    
    stats.probplot(theta, dist='norm', plot=axes[i, 1])
    axes[i, 1].set_title(f'{name}: Q-Q Plot')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'theta_distributions.png'), dpi=150)
plt.show()

for name, theta in [('Sample 1', theta1_est), ('Sample 2', theta2_est)]:
    n = len(theta)
    print(f"\n{name} (N = {n}):")
    print(f"  M = {theta.mean():.3f}, SD = {theta.std():.3f}")
    print(f"  Skewness = {stats.skew(theta):.3f}")
    print(f"  Kurtosis = {stats.kurtosis(theta):.3f}")
    # K-S test for normality (better for large N)
    ks_stat, ks_p = stats.kstest(theta, 'norm', args=(theta.mean(), theta.std()))
    print(f"  K-S test: D = {ks_stat:.4f}, p = {ks_p:.4f}")


Sample 1 (N = 449):
  M = 0.029, SD = 1.019
  Skewness = 0.372
  Kurtosis = 0.161
  K-S test: D = 0.0545, p = 0.1336

Sample 2 (N = 798):
  M = 0.000, SD = 0.976
  Skewness = 0.008
  Kurtosis = 0.044
  K-S test: D = 0.0169, p = 0.9735


---
## 4d. Scoring Method Comparison

In [5]:
def scoring_comparison(sample_full, s_auth, theta_est, author_cols, foil_cols, name):
    """Compare 4 scoring methods."""
    # Use complete-case indices matching theta estimation
    complete_mask = s_auth.dropna().index
    df = sample_full.loc[complete_mask].copy()
    
    hits = df[author_cols].sum(axis=1).values
    fa = df[foil_cols].sum(axis=1).values
    standard = hits - fa
    penalty2 = hits - 2 * fa
    theta = theta_est
    
    scores = pd.DataFrame({
        'Hits': hits,
        'Standard (H-FA)': standard,
        'Penalty2 (H-2FA)': penalty2,
        'IRT Theta (EAP)': theta
    })
    
    # Correlation matrix
    print(f"\n{'='*60}")
    print(f"Scoring Method Comparison: {name}")
    print(f"{'='*60}")
    
    methods = scores.columns
    n_m = len(methods)
    corr_matrix = np.zeros((n_m, n_m))
    p_matrix = np.zeros((n_m, n_m))
    
    for i in range(n_m):
        for j in range(n_m):
            r, p = stats.pearsonr(scores.iloc[:, i], scores.iloc[:, j])
            corr_matrix[i, j] = r
            p_matrix[i, j] = p
    
    corr_df = pd.DataFrame(corr_matrix, index=methods, columns=methods)
    print("\nPearson correlations:")
    print(corr_df.round(4).to_string())
    
    # All p-values < 0.001
    print(f"\nAll p-values: < {p_matrix[np.triu_indices(n_m, k=1)].max():.2e}")
    
    # Spearman rank correlations
    for i in range(n_m):
        for j in range(i+1, n_m):
            rho, p_s = stats.spearmanr(scores.iloc[:, i], scores.iloc[:, j])
            tau, p_t = stats.kendalltau(scores.iloc[:, i], scores.iloc[:, j])
    
    # Theta-Hits Spearman
    rho_th, p_th = stats.spearmanr(theta, hits)
    print(f"\nTheta-Hits Spearman rho = {rho_th:.4f}, p = {p_th:.2e}")
    
    return scores, corr_df

scores1, corr1 = scoring_comparison(s1_full, s1_auth, theta1_est, author_cols, foil_cols, 'Sample 1')
scores2, corr2 = scoring_comparison(s2_full, s2_auth, theta2_est, author_cols, foil_cols, 'Sample 2')


Scoring Method Comparison: Sample 1

Pearson correlations:
                    Hits  Standard (H-FA)  Penalty2 (H-2FA)  IRT Theta (EAP)
Hits              1.0000           0.9599            0.8580           0.9809
Standard (H-FA)   0.9599           1.0000            0.9676           0.9521
Penalty2 (H-2FA)  0.8580           0.9676            1.0000           0.8609
IRT Theta (EAP)   0.9809           0.9521            0.8609           1.0000

All p-values: < 2.04e-131

Theta-Hits Spearman rho = 0.9939, p = 0.00e+00



Scoring Method Comparison: Sample 2

Pearson correlations:
                    Hits  Standard (H-FA)  Penalty2 (H-2FA)  IRT Theta (EAP)
Hits              1.0000           0.8784            0.4485           0.9823
Standard (H-FA)   0.8784           1.0000            0.8211           0.9018
Penalty2 (H-2FA)  0.4485           0.8211            1.0000           0.5134
IRT Theta (EAP)   0.9823           0.9018            0.5134           1.0000

All p-values: < 9.60e-41



Theta-Hits Spearman rho = 0.9903, p = 0.00e+00


In [6]:
# Scatter plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row, (scores, name) in enumerate([(scores1, 'Sample 1'), (scores2, 'Sample 2')]):
    theta = scores['IRT Theta (EAP)'].values
    hits = scores['Hits'].values
    std = scores['Standard (H-FA)'].values
    
    axes[row, 0].scatter(hits, theta, alpha=0.3, s=10)
    axes[row, 0].set_xlabel('Hits')
    axes[row, 0].set_ylabel('IRT Theta')
    axes[row, 0].set_title(f'{name}: Hits vs Theta')
    
    axes[row, 1].scatter(std, theta, alpha=0.3, s=10)
    axes[row, 1].set_xlabel('Standard Score')
    axes[row, 1].set_ylabel('IRT Theta')
    axes[row, 1].set_title(f'{name}: Standard vs Theta')
    
    # Bland-Altman: Theta vs standardized Hits
    hits_z = (hits - hits.mean()) / hits.std()
    diff = theta - hits_z
    mean_both = (theta + hits_z) / 2
    axes[row, 2].scatter(mean_both, diff, alpha=0.3, s=10)
    axes[row, 2].axhline(diff.mean(), color='red', linestyle='-')
    axes[row, 2].axhline(diff.mean() + 1.96*diff.std(), color='red', linestyle='--')
    axes[row, 2].axhline(diff.mean() - 1.96*diff.std(), color='red', linestyle='--')
    axes[row, 2].set_xlabel('Mean of Theta & z(Hits)')
    axes[row, 2].set_ylabel('Difference')
    axes[row, 2].set_title(f'{name}: Bland-Altman')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'scoring_comparison.png'), dpi=150)
plt.show()

In [7]:
# Save participant-level scores
scores1.to_csv(os.path.join(RESULTS_DIR, 'participant_scores_sample1.csv'), index=False)
scores2.to_csv(os.path.join(RESULTS_DIR, 'participant_scores_sample2.csv'), index=False)

# Save IRT marginal reliability
for name, tif in [('Sample 1', tif1), ('Sample 2', tif2)]:
    marginal_rel = 1 - len(a1) / tif.sum()
    print(f"{name} IRT marginal reliability: {marginal_rel:.4f}")

print(f"\nFiles saved to: {RESULTS_DIR}")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {f}")

Sample 1 IRT marginal reliability: 0.9751
Sample 2 IRT marginal reliability: 0.9712

Files saved to: /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/data/processed/results/04_test_ability
  participant_scores_sample1.csv
  participant_scores_sample2.csv
  scoring_comparison.png
  theta_distributions.png
  tif_sem_reliability.png
